In [ ]:
# Install dependencies
!apt-get install -y gmt ghostscript
!pip install pygmt==0.14.0 obspy pandas matplotlib

In [ ]:
# Import libraries
import os

!ls -l /usr/lib/x86_64-linux-gnu/libgmt*

!ln -sfn /usr/lib/x86_64-linux-gnu/libgmt.so.6.4.0 /usr/lib/x86_64-linux-gnu/libgmt.so

# Set the GMT_LIBRARY_PATH environment variable
os.environ["GMT_LIBRARY_PATH"] = "/usr/lib/x86_64-linux-gnu/"
import pygmt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import os

!ln -sfn /usr/lib/x86_64-linux-gnu/libgmt.so.6.4.0 /usr/lib/x86_64-linux-gnu/libgmt.so

os.environ["GMT_LIBRARY_PATH"] = "/usr/lib/x86_64-linux-gnu/"
import pygmt
import pandas as pd

# ---------------------------
# LOAD DATA
# ---------------------------
df = pd.read_csv("/content/drive/MyDrive/Earthquake data/IEB_export (1).csv")

# --- Construct full datetime from separate columns ---
df['datetime_str'] = df['Year'].astype(str) + '-' + \
                     df['Month'].astype(str) + '-' + \
                     df['Day'].astype(str) + ' ' + \
                     df['Time'].astype(str)
df['time'] = pd.to_datetime(df['datetime_str'], errors='coerce', utc=True)

# Rename other columns
df = df.rename(columns={
    "Lat": "lat",
    "Lon": "lon",
    "Depth": "depth",
    "Mag": "mag"
})

df = df.dropna(subset=["lat", "lon", "depth", "mag", "time"])

# --- Diagnostic print statements ---
print(f"Full data time range in loaded CSV: {df['time'].min()} to {df['time'].max()}")
print("Head of the DataFrame before filtering:")
print(df.head())
# -----------------------------------

# ---------------------------
# MAINSHOCK INFO
# ---------------------------
main_lat = 24.834
main_lon = 93.656
main_time = pd.to_datetime("2016-01-04", utc=True)

# ---------------------------
# FILTER DATA
# ---------------------------

# Time ±100 days
df = df[
    (df["time"] >= main_time - pd.Timedelta(days=100)) &
    (df["time"] <= main_time + pd.Timedelta(days=100))
]

# Region 20° × 20°
df = df[
    (df["lat"] >= main_lat - 10) & (df["lat"] <= main_lat + 10) &
    (df["lon"] >= main_lon - 10) & (df["lon"] <= main_lon + 10)
]

# Magnitude >= 0 (updated)
df = df[df["mag"] >= 0]

print("Filtered events:", len(df))

# --- Conditional plotting based on df content ---
if df.empty:
    print("Cannot plot: DataFrame is empty after filtering. Please adjust filters or data.")
else:
    # ---------------------------
    # SIZE (CIRCLE BASED ON MAG)
    # ---------------------------
    df["size"] = 0.5 * (2 ** (df["mag"] - 5))
    # ---------------------------
    # REGION
    # ---------------------------
    region = [
        main_lon - 10, main_lon + 10,
        main_lat - 10, main_lat + 10
    ]

    # ---------------------------
    # FIGURE (YOUR BASEMAP STYLE)
    # ---------------------------
    fig = pygmt.Figure()

    fig.grdimage(
        grid="@earth_relief_15s",
        region=region,
        projection="M6i",
        shading=True,
        cmap="geo",

        frame=["af", "WSne+t\"Earthquake Distribution Map (Imphal Region)\""]
)
# Coastlines & borders
    fig.coast(
        borders="1/0.5p,black",
        shorelines="1/0.5p,black"

    )

    # ---------------------------
    # DEPTH COLOR
    # ---------------------------
    pygmt.makecpt(cmap="viridis", series=[0, 150])

    # ---------------------------
    # DEPTH COLORBAR (BOTTOM)
    # ---------------------------
    fig.colorbar(
        position="JBC+w12c/0.6c+h+o0c/0.8c",
        frame='af+l"Depth (km)"'
    )

    # ---------------------------
    # ELEVATION COLORBAR (ABOVE)
    # ---------------------------
    fig.colorbar(
        cmap="geo",
        position="JBC+w12c/0.6c+h+o0c/2.6c",
        frame='af+l"Elevation (m)"'
    )


    # ---------------------------
    # PLOT EARTHQUAKES (CIRCLES)
    # ---------------------------
    fig.plot(
        x=df["lon"],
        y=df["lat"],
        size=df["size"],
        fill=df["depth"],
        cmap=True,
        style="c",
        pen="black"
    )

    # ---------------------------
    # MAINSHOCK STAR
    # ---------------------------
    fig.plot(
        x=main_lon,
        y=main_lat,
        style="a0.6c",
        fill="red",  # Changed 'color' to 'fill'
        pen="black"
    )



    # ---------------------------
# MAGNITUDE LEGEND (MIN → MAX, 4 CIRCLES)
# ---------------------------

    legend_x = region[1] - 3
    legend_y = region[3] - 2

    fig.text(
    x=legend_x,
    y=legend_y + 1,
    text="Magnitude (Mw)",
    font="12p,Helvetica-Bold"
)

# Get min and max magnitude
    mag_min = df["mag"].min()
    mag_max = df["mag"].max()

# Create 4 evenly spaced magnitudes
    mags = [
    mag_min,
    mag_min + (mag_max - mag_min)/3,
    mag_min + 2*(mag_max - mag_min)/3,
    mag_max
]

# Plot legend
    for i, m in enumerate(mags):
        size = 0.5 * (2 ** (m - 5))   # SAME scaling as map

        y_pos = legend_y - i * 1.5

        fig.plot(
            x=legend_x,
            y=y_pos,
            style=f"c{size}c",
            fill="white",
            pen="black"
        )

        fig.text(
            x=legend_x + 1.5,
            y=y_pos,
            text=f"Mw {m:.1f}",
            font="10p,Helvetica"
        )
    # Dynamic spacing (KEY FIX)
        y_pos -= (size + 0.4)
    # 9. IMPROVED INSET MAP (CLEAN)
    # -----------------------------
    with fig.inset(position="jTL+w3.8c+o0.4c", box="+p1p,black+gwhite@90"):

        # Globe centered exactly at study area
        fig.coast(
            region="g",                         # global
            projection=f"G{main_lon}/{main_lat}/3.8c",    # center at Imphal
            land="green",
            water="skyBlue",
            shorelines="0.4p,black",
            borders="1/0.3p,black"
        )

        # Study area box (thin & neat)
        fig.plot(
            x=[region[0], region[1], region[1], region[0], region[0]],
            y=[region[2], region[2], region[3], region[3], region[2]],
            pen="1.5p,red"
        )

        # Epicenter (clear but small)
        fig.plot(
            x=main_lon,
            y=main_lat,
            style="c0.12c",
            fill="red",
            pen="black"
        )
    fig.savefig("final_overlay_map.pdf")
    fig.show()

In [ ]:
import pygmt

# -----------------------------
# 1. Mainshock details
# -----------------------------
lat = 24.834
lon = 93.656

# 20° × 20° region
region = [lon - 10, lon + 10, lat - 10, lat + 10]

# -----------------------------
# 2. Load Earth relief
# -----------------------------
grid = pygmt.datasets.load_earth_relief(resolution="15s", region=region)

# -----------------------------
# 3. Create figure
# -----------------------------
fig = pygmt.Figure()

# -----------------------------
# 4. Plot topography
# -----------------------------
fig.grdimage(
    grid=grid,
    projection="M6i",
    cmap="geo",
    shading=True
)

# Coastlines & borders
fig.coast(
    borders="1/0.5p,black",
    shorelines="1/0.5p,black",
    frame=["af", "+tImphal Earthquake (2016)"]
)

# -----------------------------
# 5. Plot mainshock
# -----------------------------
fig.plot(
    x=lon,
    y=lat,
    style="a0.7c",
    fill="red", # Changed 'color' to 'fill'
    pen="black"
)

fig.text(
    x=96.80,
    y=24.95,
    text="Mainshock(Mw 6.7)",
    font="12p,Helvetica-Bold,black"
)

# -----------------------------
# 6. Cities
# -----------------------------
cities = [
    ("Guwahati", 26.1445, 91.7362),
    ("Aizawl", 23.7271, 92.7176),
    ("Kolkata", 22.5726, 88.3639),
    ("Dhaka", 23.8103, 90.4125),
    ("Mandalay", 21.9588, 96.0891),
    ("Naypyidaw", 19.7633, 96.0785)
]

for name, lat_c, lon_c in cities:
    fig.plot(x=lon_c, y=lat_c, style="c0.2c", fill="black") # Increased city marker size
    fig.text(x=lon_c + 0.4, y=lat_c + 0.4, text=name, font="12p,black") # Made text bigger

# -----------------------------
# 7. Colorbar
# -----------------------------
fig.colorbar(frame=["a2000", "x+lElevation (m)"])

# -----------------------------
# 8. Scale bar & north arrow
# -----------------------------
fig.basemap(map_scale="jBL+w500k+o0.5c/0.5c")
fig.basemap(rose="jTR+w2c+f")

# 9. IMPROVED INSET MAP (CLEAN)
# -----------------------------
with fig.inset(position="jTL+w3.8c+o0.4c", box="+p1p,black+gwhite@90"):

    # Globe centered exactly at study area
    fig.coast(
        region="g",                         # global
        projection=f"G{lon}/{lat}/3.8c",    # center at Imphal
        land="green",
        water="skyBlue",
        shorelines="0.4p,black",
        borders="1/0.3p,black"
    )

    # Study area box (thin & neat)
    fig.plot(
        x=[region[0], region[1], region[1], region[0], region[0]],
        y=[region[2], region[2], region[3], region[3], region[2]],
        pen="1.5p,red"
    )

    # Epicenter (clear but small)
    fig.plot(
        x=lon,
        y=lat,
        style="c0.12c",
        fill="red",
        pen="black"
    )
# -----------------------------
# LEGEND (Clean)
# -----------------------------
with open("legend.txt", "w") as f:
    f.write("H 10 Legend\n")
    f.write("S 0.3c a 0.5c red 0.25p 0.8c Mainshock\n")
    f.write("S 0.3c c 0.2c black 0.25p 0.8c Cities\n")


fig.legend(
    spec="legend.txt",
    position="JBR+jBR+o0.3c",
    box="+p1p+gwhite"
)



# -----------------------------
# 11. Save & show
# -----------------------------
fig.savefig("imphal_earthquake_map.pdf", dpi=600)
fig.show()


In [ ]:
import pygmt
import pandas as pd

# ----------------------------------------------------
# Read GCMT data (ONLY required columns)
# ----------------------------------------------------
cols = ["lon", "lat", "depth", "mrr", "mtt", "mpp", "mrt", "mrp", "mtp"]

gcmt = pd.read_csv(
    "/content/drive/MyDrive/Earthquake data/focal data.txt",
    delim_whitespace=True,
    header=None,
    usecols=range(9),   # <-- THIS FIXES THE ERROR
    names=cols
)

In [ ]:
region = [85.127, 105.127, 15.149, 35.149]

# Define lon and lat for the inset map, consistent with previous cells' mainshock location
lon = 93.656
lat = 24.834

fig = pygmt.Figure()

fig.basemap(
    region=region,
    projection="M15c",
    frame=["a", "+tGCMT Focal Mechanisms (Imphal Region)"]
)



fig.basemap(map_scale="jBL+o0.5c/0.5c+w200k")

fig.basemap(rose="jTR+w2.5c")


# Topography
fig.grdimage(
    grid="@earth_relief_30s",
    shading=True,
    cmap="geo"
)

# Coastlines & borders
fig.coast(
    shorelines="0.8p,black",
    borders="1/0.6p,black"
)


# Rename columns to match PyGMT's expected conventions for meca
gcmt_processed = gcmt.rename(columns={'lon': 'longitude', 'lat': 'latitude'})

# Add a default 'exponent' column (common default for seismic moment in dyne-cm is 20)
gcmt_processed['exponent'] = 20

# Ensure the DataFrame only contains and orders the expected columns for 'mt' convention
expected_meca_cols = ['longitude', 'latitude', 'depth', 'mrr', 'mtt', 'mpp', 'mrt', 'mrp', 'mtp', 'exponent']
gcmt_processed = gcmt_processed[expected_meca_cols]

# Save the DataFrame to a temporary CSV file and use it as input
tmp_gcmt_file = "gcmt_meca_NEW.csv"
gcmt_processed.to_csv(tmp_gcmt_file, sep=' ', header=False, index=False)

fig.meca(
    spec=tmp_gcmt_file, # Pass the file path as input
    convention="mt",      # moment tensor
    scale="0.8c",
    compressionfill="black",
    extensionfill="white",
    pen="0.6p,black"
)

# 9. IMPROVED INSET MAP (CLEAN)
# -----------------------------
with fig.inset(position="jTL+w3.8c+o0.4c", box="+p1p,black+gwhite@90"):

    # Globe centered exactly at study area
    fig.coast(
        region="g",
        projection=f"G{lon}/{lat}/3.8c",
        land="green",
        water="skyBlue",
        shorelines="0.4p,black",
        borders="1/0.3p,black"
    )

    # Study area box (thin & neat)
    fig.plot(
        x=[region[0], region[1], region[1], region[0], region[0]],
        y=[region[2], region[2], region[3], region[3], region[2]],
        pen="1.5p,red"
    )

    # Epicenter (clear but small)
    fig.plot(
        x=lon,
        y=lat,
        style="c0.12c",
        fill="red",
        pen="black"
    )
fig.basemap(map_scale="jBL+w500k+o0.5c/0.5c")
fig.basemap(rose="jTR+w2c+f")
#  Colorbar
# -----------------------------
fig.colorbar(frame=["a2000", "x+lElevation (m)"])
# ----------------------------------------------------
# Save figure
# ----------------------------------------------------
fig.savefig("Figure2c_GCMT.pdf")
fig

In [ ]:
import pygmt

# -----------------------------
# 1. Mainshock details
# -----------------------------
lat = 24.834
lon = 93.656

# 20° × 20° region
region = [lon - 10, lon + 10, lat - 10, lat + 10]

# -----------------------------
# 2. Load Earth relief
# -----------------------------
grid = pygmt.datasets.load_earth_relief(resolution="15s", region=region)

# -----------------------------
# 3. Create figure
# -----------------------------
fig = pygmt.Figure()

# -----------------------------
# 4. Plot topography
# -----------------------------
fig.grdimage(
    grid=grid,
    projection="M6i",
    cmap="geo",
    shading=True
)

# Coastlines & borders
fig.coast(
    borders="1/0.5p,black",
    shorelines="1/0.5p,black",
    frame=["af", "+tImphal Earthquake (2016)"]
)

# -----------------------------
# 5. Plot mainshock
# -----------------------------
fig.plot(
    x=lon,
    y=lat,
    style="a0.7c",
    fill="red", # Changed 'color' to 'fill'
    pen="black"
)

fig.text(
    x=96.80,
    y=24.95,
    text="Mainshock(Mw 6.7)",
    font="12p,Helvetica-Bold,black"
)

# -----------------------------
# 6. Cities
# -----------------------------
cities = [
    ("Guwahati", 26.1445, 91.7362),
    ("Aizawl", 23.7271, 92.7176),
    ("Kolkata", 22.5726, 88.3639),
    ("Dhaka", 23.8103, 90.4125),
    ("Mandalay", 21.9588, 96.0891),
    ("Naypyidaw", 19.7633, 96.0785)
]

for name, lat_c, lon_c in cities:
    fig.plot(x=lon_c, y=lat_c, style="c0.2c", fill="black") # Increased city marker size
    fig.text(x=lon_c + 0.4, y=lat_c + 0.4, text=name, font="12p,black") # Made text bigger

# -----------------------------
# 7. Colorbar
# -----------------------------
fig.colorbar(frame=["a2000", "x+lElevation (m)"])

# -----------------------------
# 8. Scale bar & north arrow
# -----------------------------
fig.basemap(map_scale="jBL+w500k+o0.5c/0.5c")
fig.basemap(rose="jTR+w2c+f")

# 9. IMPROVED INSET MAP (CLEAN)
# -----------------------------
with fig.inset(position="jTL+w3.8c+o0.4c", box="+p1p,black+gwhite@90"):

    # Globe centered exactly at study area
    fig.coast(
        region="g",                         # global
        projection=f"G{lon}/{lat}/3.8c",    # center at Imphal
        land="green",
        water="skyBlue",
        shorelines="0.4p,black",
        borders="1/0.3p,black"
    )

    # Study area box (thin & neat)
    fig.plot(
        x=[region[0], region[1], region[1], region[0], region[0]],
        y=[region[2], region[2], region[3], region[3], region[2]],
        pen="1.5p,red"
    )

    # Epicenter (clear but small)
    fig.plot(
        x=lon,
        y=lat,
        style="c0.12c",
        fill="red",
        pen="black"
    )
# -----------------------------
# LEGEND (Clean)
# -----------------------------
with open("legend.txt", "w") as f:
    f.write("H 10 Legend\n")
    f.write("S 0.3c a 0.5c red 0.25p 0.8c Mainshock\n")
    f.write("S 0.3c c 0.2c black 0.25p 0.8c Cities\n")


fig.legend(
    spec="legend.txt",
    position="JBR+jBR+o0.3c",
    box="+p1p+gwhite"
)



# -----------------------------
# 11. Save & show
# -----------------------------
fig.savefig("imphal_earthquake_map.pdf", dpi=600)
fig.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from scipy.optimize import curve_fit

df = pd.read_csv("/content/drive/MyDrive/Gaurav Dev Assignment 2/IEB_export (1).csv")

# CORRECTED: Parse full datetime from separate columns
df["Datetime"] = pd.to_datetime(
    df["Year"].astype(str) + "-" +
    df["Month"].astype(str) + "-" +
    df["Day"].astype(str) + " " +
    df["Time"],
    errors='coerce',
    utc=True
)

# Drop rows where datetime parsing failed
df = df.dropna(subset=["Datetime"])

mainshock_time = pd.to_datetime("2016-01-03T23:05:22", utc=True)

# Filter for foreshocks and aftershocks based on the corrected mainshock time
foreshocks = df[df["Datetime"] < mainshock_time].copy()
aftershocks = df[df["Datetime"] > mainshock_time].copy()

foreshocks = foreshocks[foreshocks["Mag"] >= 3.0]
aftershocks = aftershocks[aftershocks["Mag"] >= 3.0]

foreshocks["t_days"] = (
    mainshock_time - foreshocks["Datetime"]).dt.total_seconds() / 86400

aftershocks["t_days"] = (
    aftershocks["Datetime"] - mainshock_time).dt.total_seconds() / 86400

# Ensure aftershocks have positive t_days
aftershocks = aftershocks[aftershocks["t_days"] > 0]

# Check if there are enough aftershocks to proceed
if aftershocks.empty:
    print("No aftershocks found for fitting after filtering. Please check mainshock time or data range.")
else:
    min_t_days = aftershocks["t_days"].min()
    if min_t_days <= 0:
        min_t_days = 1e-6

    # Adjust the number of bins dynamically or keep it reasonable
    num_bins = min(50, int(aftershocks["t_days"].max())) if aftershocks["t_days"].max() > 1 else 10
    if aftershocks["t_days"].max() == 0:
      num_bins = 10

    bins = np.logspace(np.log10(min_t_days), np.log10(aftershocks["t_days"].max()), num_bins)

    counts, edges = np.histogram(aftershocks["t_days"], bins=bins)
    dt = edges[1:] - edges[:-1]

    # Filter out bins with zero counts or zero width
    mask_bins = (counts > 0) & (dt > 0)
    rate = counts[mask_bins] / dt[mask_bins]
    t_mid = ((edges[:-1] + edges[1:]) / 2)[mask_bins]


    if len(t_mid) < 3: # Need at least 3 points for curve_fit with 3 parameters
        print("Not enough data points after binning and filtering for Omori law fitting.")
    else:
        def omori_law(t, K, c, p):
            return K / (t + c)**p
        p0 = [rate.max(), 0.1, 1.0] # Initial guess

        try:
            # Add bounds to ensure parameters are positive, which is physically realistic for Omori's law
            params, _ = curve_fit(omori_law, t_mid, rate, p0=p0, maxfev=10000, bounds=(0, np.inf))
            K, c, p = params

            t_fit = np.linspace(t_mid.min(), t_mid.max(), 500)
            rate_fit = omori_law(t_fit, K, c, p)

            fig, axs = plt.subplots(1, 2, figsize=(12, 5))

            axs[0].scatter(t_mid, rate,color="black",s=35,label="Observed rate")

            axs[0].plot( t_fit, rate_fit,color="red",linewidth=2.5,
                label=f"Omori law (p = {p:.2f})")

            axs[0].set_xlabel("Time after mainshock (days)")
            axs[0].set_ylabel("Aftershock rate (events/day)")
            axs[0].set_title("Aftershocks (Mw ≥ 3.0) – Omori Decay")

            axs[0].grid(True)
            axs[0].legend()

            axs[1].scatter(foreshocks["t_days"],foreshocks["Mag"],color="blue",s=35)

            axs[1].set_xlabel("Time before mainshock (days)")
            axs[1].set_ylabel("Magnitude")
            axs[1].set_title("Foreshocks (Mw ≥ 3.0)")
            axs[1].grid(True)

            plt.tight_layout()
            plt.savefig("Figure2e.pdf", dpi=300)
            plt.show()

        except RuntimeError as e:
            print(f"curve_fit failed: {e}")
            print("It's possible that the data points are too sparse or noisy for a good fit, even after corrections.")
            print(f"Number of data points for fit: {len(t_mid)}")
            if len(t_mid) > 0:
                print(f"t_mid range: [{t_mid.min():.2f}, {t_mid.max():.2f}]")
                print(f"rate range: [{rate.min():.2f}, {rate.max():.2f}]")
            else:
                print("No valid t_mid and rate data for fitting.")


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# ------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------
df = pd.read_csv("/content/drive/MyDrive/Earthquake data/IEB_export (1).csv")

# Use magnitude column
magnitudes = df["Mag"].dropna()

# ------------------------------------------------
# 2. SORT MAGNITUDES
# ------------------------------------------------
magnitudes = np.sort(magnitudes)

# ------------------------------------------------
# 3. CUMULATIVE NUMBER (N ≥ M)
# ------------------------------------------------
M = np.sort(np.unique(magnitudes))
N = np.array([np.sum(magnitudes >= m) for m in M])


# Remove zeros
mask = N > 0
M = M[mask]
N = N[mask]

logN = np.log10(N)

# ------------------------------------------------
# 4. FIT GUTENBERG–RICHTER LAW
# log10(N) = a - bM
# ------------------------------------------------
slope, intercept, r_value, _, _ = linregress(M, logN)

b_value = -slope
a_value = intercept

# ------------------------------------------------
# 5. FREQUENCY (HISTOGRAM)
# ------------------------------------------------
bins = np.arange(min(magnitudes), max(magnitudes) + 0.1, 0.1)
freq, edges = np.histogram(magnitudes, bins=bins)

M_mid = (edges[:-1] + edges[1:]) / 2

# ------------------------------------------------
# 6. PLOTTING
# ------------------------------------------------
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# ---- LEFT: Magnitude vs log(N) ----
axs[0].scatter(M, logN, color="blue", label="Observed")

# Fit line
M_fit = np.linspace(min(M), max(M), 100)
logN_fit = intercept + slope * M_fit

axs[0].plot(M_fit, logN_fit, 'r',
            label=f"Fit: b = {b_value:.2f}, R²={r_value**2:.2f}")

axs[0].set_xlabel("Magnitude (M)")
axs[0].set_ylabel("log₁₀(N ≥ M)")
axs[0].set_title("Gutenberg–Richter Law")
axs[0].legend()
axs[0].grid(True)

# ---- RIGHT: Magnitude vs Frequency ----
axs[1].bar(M_mid, freq, width=0.08, color="green", edgecolor="black")

axs[1].set_xlabel("Magnitude (M)")
axs[1].set_ylabel("Frequency")
axs[1].set_title("Magnitude vs Frequency")
axs[1].grid(True)

plt.tight_layout()
plt.savefig("Gutenberg_Richter_Plots.pdf", dpi=300)
plt.show()

# ------------------------------------------------
# 7. PRINT RESULTS
# ------------------------------------------------
print("----- GUTENBERG–RICHTER RESULTS -----")
print(f"a-value = {a_value:.3f}")
print(f"b-value = {b_value:.3f}")
print(f"R² = {r_value**2:.3f}")

In [ ]:
import numpy as np
import pandas as pd
import pygmt

# FILES
FILES = {
    "BHE": "/content/drive/MyDrive/Earthquake data/HK.HKPS..BHE.M.2016-01-03T230738.019500 (1).csv",
    "BHN": "/content/drive/MyDrive/Earthquake data/HK.HKPS..BHN.M.2016-01-03T230738.019500.csv",
    "BHZ": "/content/drive/MyDrive/Earthquake data/HK.HKPS..BHZ.M.2016-01-03T230738.019500.csv"
}

SAMPLE_RATE = 10.0

COMP_ORDER  = ["BHZ", "BHN", "BHE"]
COMP_LABEL  = ["BHZ (Vertical)", "BHN (North)", "BHE (East)"]
COMP_COLOR  = ["firebrick", "royalblue", "darkgreen"]

# READ DATA
raw = {}
norms = {}

for comp, fname in FILES.items():
    samples = pd.read_csv(fname, comment="#")["Sample"].values.astype(float)
    samples -= np.mean(samples)
    raw[comp] = samples

    peak = np.max(np.abs(samples))
    norms[comp] = samples / peak if peak > 0 else samples

n_samples = len(raw["BHZ"])
dt = 1.0 / SAMPLE_RATE
time_min = np.arange(n_samples) * dt / 60.0

# FIGURE SETTINGS
W, H = 16, 3.8
xL, yB, GAP = 2.8, 1.8, 0.7
Y_PANEL = [yB + k * (H + GAP) for k in range(3)]

t0, t1 = time_min[0], time_min[-1]

fig = pygmt.Figure()

for row, (comp, label, color) in enumerate(zip(COMP_ORDER, COMP_LABEL, COMP_COLOR)):
    panel_y = Y_PANEL[2 - row]

    region = [t0, t1, -1.15, 1.15]

    x_frame = ('xa5f1+l"Time (minutes)"' if row == 2 else "xf1")
    y_frame = f'ya0.5f0.25+l"{label}"'

    sides = ("WNe" if row == 0 else "WSe" if row == 2 else "We")

    fig.shift_origin(xshift=f"{xL}c", yshift=f"{panel_y}c")

    fig.basemap(region=region, projection=f"X{W}c/{H}c",
                frame=[sides, x_frame, y_frame])

    fig.plot(x=[t0, t1], y=[0, 0], pen="0.4p,gray,-")
    fig.plot(x=time_min, y=norms[comp], pen=f"0.6p,{color}")

    fig.text(x=t1*0.98, y=0.9, text=comp,
             font=f"10p,Helvetica-Bold,{color}", justify="TR")

    fig.shift_origin(xshift=f"-{xL}c", yshift=f"-{panel_y}c")
fig.savefig("waveform data.pdf")
# ✅ SHOW ONCE
fig.show()


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------
# 1. LOAD YOUR FILE (UPLOAD FIRST)
# ------------------------------------------------
file_path = "/content/drive/MyDrive/Earthquake data/IEB_export (1).csv"
df = pd.read_csv(file_path)

# ------------------------------------------------
# 2. CREATE DATETIME COLUMN
# ------------------------------------------------
df["Datetime"] = pd.to_datetime(
    df["Year"].astype(str) + "-" +
    df["Month"].astype(str) + "-" +
    df["Day"].astype(str) + " " +
    df["Time"]
)

# ------------------------------------------------
# 3. IDENTIFY MAINSHOCK (MAX MAGNITUDE)
# ------------------------------------------------
mainshock = df.loc[df["Mag"].idxmax()]
mainshock_time = mainshock["Datetime"]
M_main = mainshock["Mag"]

# ------------------------------------------------
# 4. EXTRACT AFTERSHOCKS
# ------------------------------------------------
aftershocks = df[df["Datetime"] > mainshock_time].copy()

# Remove very small magnitudes if needed
aftershocks = aftershocks[aftershocks["Mag"] > 0]

# ------------------------------------------------
# 5. LARGEST AFTERSHOCK
# ------------------------------------------------
M_after_max = aftershocks["Mag"].max()

# ------------------------------------------------
# 6. COMPUTE ΔM
# ------------------------------------------------
delta_M = M_main - M_after_max

# ------------------------------------------------
# 7. PLOT FIGURE (SUBMISSION QUALITY)
# ------------------------------------------------
plt.figure(figsize=(6,5))

bars = plt.bar(
    ["Mainshock", "Largest Aftershock"],
    [M_main, M_after_max]
)

# Add values on bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2,
             height + 0.05,
             f"{height:.2f}",
             ha='center', fontsize=10)

plt.ylabel("Magnitude")
plt.title(f"Bath's Law Analysis (ΔM = {delta_M:.2f})")

plt.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig("Baths_Law_Figure.pdf", dpi=300)
plt.show()

# ------------------------------------------------
# 8. PRINT RESULT
# ------------------------------------------------
print("----- BATH'S LAW RESULT -----")
print(f"Mainshock Magnitude = {M_main:.2f}")
print(f"Largest Aftershock = {M_after_max:.2f}")
print(f"ΔM = {delta_M:.2f}")

# Interpretation
if abs(delta_M - 1.2) <= 0.3:
    print("✔ Bath's law is approximately satisfied.")
else:
    print("✘ Bath's law is not well satisfied.")

In [ ]:
# ============================================
# SEISMIC CROSS SECTION (LEGEND OUTSIDE)
# ============================================

import numpy as np
import pandas as pd
import pygmt
from pyproj import Geod

# --------------------------------------------
# 1. LOAD DATA
# --------------------------------------------
df = pd.read_csv("/content/drive/MyDrive/Earthquake data/IEB_export (1).csv")

df = df.rename(columns={
    "Lat": "lat",
    "Lon": "lon",
    "Depth": "depth",
    "Mag": "mag"
})

print("Total earthquakes:", len(df))

# --------------------------------------------
# 2. PROFILE
# --------------------------------------------
start = (91.5, 24.0)
end   = (96.5, 25.5)

geod = Geod(ellps="WGS84")

def project_point(lon, lat):
    lon1, lat1 = start
    lon2, lat2 = end

    az12, _, _ = geod.inv(lon1, lat1, lon2, lat2)
    az13, _, dist = geod.inv(lon1, lat1, lon, lat)

    angle = np.radians(az13 - az12)
    return (dist * np.cos(angle)) / 1000

df["distance"] = df.apply(lambda r: project_point(r["lon"], r["lat"]), axis=1)

# --------------------------------------------
# 3. SLAB
# --------------------------------------------
x = np.linspace(0, 800, 400)
slab = 30 + 0.0009*(x-100)**2 + 0.08*x
upper = slab - 40
lower = slab + 40

# --------------------------------------------
# 4. FIGURE (EXTENDED REGION)
# --------------------------------------------
fig = pygmt.Figure()

region = [0, 650, 0, 200]   # 👈 extended to right
projection = "X18c/-10c"

fig.basemap(
    region=region,
    projection=projection,
    frame=[
        "xaf+lDistance (km)",
        "yaf+lDepth (km)",
        "WSen+tSeismic Cross Section"
    ]
)

# --------------------------------------------
# 5. COLOR MAP
# --------------------------------------------
pygmt.makecpt(cmap="viridis", series=[0, 200])

# --------------------------------------------
# 6. SIZE BY MAGNITUDE
# --------------------------------------------
scale_factor = 0.18
df["size"] = 0.15 + (df["mag"] - df["mag"].min()) * scale_factor

# --------------------------------------------
# 7. PLOT DATA (ONLY 0–500 km)
# --------------------------------------------
fig.plot(
    x=df["distance"],
    y=df["depth"],
    style="cc",
    size=df["size"],
    fill=df["depth"],
    cmap=True,
    pen="0.2p,black"
)

# --------------------------------------------
# 8. SLAB
# --------------------------------------------
fig.plot(x=x, y=slab, pen="2.5p,blue")
fig.plot(x=x, y=upper, pen="1p,black,--")
fig.plot(x=x, y=lower, pen="1p,black,--")

# --------------------------------------------
# 9. LABELS
# --------------------------------------------

fig.text(x=80, y=25, text="Indian Plate", font="12p,Helvetica-Bold")
fig.text(x=480, y=25, text="Burma Plate", font="12p,Helvetica-Bold")

# --------------------------------------------
# 10. COLORBAR
# --------------------------------------------
fig.colorbar(frame="af+lDepth (km)")

# --------------------------------------------
# 11. LEGEND OUTSIDE (RIGHT SIDE)
# --------------------------------------------
legend_mags = [3.5, 4.5, 5.5, 6.5]

x_leg = 535   # 👈 outside data region
y_leg = 40

fig.text(
    x=x_leg,
    y=y_leg - 20,
    text="Magnitude (Mw)",
    font="12p,Helvetica-Bold",
    justify="LM"
)

for i, m in enumerate(legend_mags):

    size = 0.15 + (m - df["mag"].min()) * scale_factor
    y_pos = y_leg + i * 25

    fig.plot(
        x=[x_leg],
        y=[y_pos],
        style="cc",
        size=[size],
        fill="white",
        pen="1p,black"
    )

    fig.text(
        x=x_leg + 30,
        y=y_pos,
        text=f"Mw {m}",
        font="10p,black",
        justify="LM"
    )

# --------------------------------------------
# 12. SAVE
# --------------------------------------------
fig.savefig("cross_section_legend_outside.pdf", dpi=300)

fig.show()

In [ ]:
import pandas as pd
import numpy as np # Added for np.nan used in magnitude parsing

# -----------------------------
# 1. LOAD FILE
# -----------------------------
file_path = "/content/Official Website of National Center of Seismology (2).xlsx"
df = pd.read_excel(file_path, header=1) #

df.columns = df.columns.str.strip()

# Detect columns using more robust names
date_time_col = [c for c in df.columns if 'Origin Time' in c][0]
lat_col  = [c for c in df.columns if 'Lat' in c][0]
lon_col  = [c for c in df.columns if 'Long' in c][0]
mag_col  = [c for c in df.columns if 'Magnitude' in c][0]

# -----------------------------
# 2. DATETIME & MAGNITUDE PARSING
# -----------------------------
df['datetime_str'] = df[date_time_col].astype(str).str.replace(' IST', '', regex=False)
df['datetime'] = pd.to_datetime(
    df['datetime_str'],
    utc=True, errors='coerce'
)

df = df.dropna(subset=['datetime'])

# Rename columns for consistency
df = df.rename(columns={
    lat_col: 'lat',
    lon_col: 'lon',
    mag_col: 'mag'
})

# Extract numerical magnitude from 'mag' column (e.g., "3.3[ML]")
df['mag'] = df['mag'].astype(str).str.extract('([0-9.]+)', expand=False).astype(float)

# -----------------------------
# 3. DEFINE MAINSHOCK
# -----------------------------
mainshock_time = pd.to_datetime('2016-01-04T23:05:22', utc=True)


mainshock_candidates = df[
    (df['datetime'] >= mainshock_time - pd.Timedelta(hours=1))
    & (df['datetime'] <= mainshock_time + pd.Timedelta(hours=1))
    & (df['mag'] >= 6.5)
]

if not mainshock_candidates.empty:
    mainshock = mainshock_candidates.loc[mainshock_candidates['mag'].idxmax()]
    M_main = mainshock['mag']
else:
    print("Warning: Mainshock not found in data or its magnitude/time. Using assumed M_main = 6.7")
    M_main = 6.7


print("Mainshock Magnitude:", M_main)

# -----------------------------
# 4. AFTERSHOCKS (30 days)
# -----------------------------
df['time_diff'] = (df['datetime'] - mainshock_time).dt.total_seconds() / (3600*24)

aftershocks = df[
    (df['time_diff'] > 0)
    & (df['time_diff'] <= 30)
    & (df['mag'] >= 0)
].copy()

# Check if aftershocks DataFrame is empty
if aftershocks.empty:
    print("No aftershocks found within the 30-day window and magnitude filter.")
    M_after = np.nan # Set to NaN if no aftershocks
else:
    # -----------------------------
    # 5. LARGEST AFTERSHOCK
    # -----------------------------
    M_after = aftershocks['mag'].max()

print("Largest Aftershock Magnitude:", M_after)

# -----------------------------
# 6. BATH'S LAW
# -----------------------------
if not np.isnan(M_after):
    delta_M = M_main - M_after
    print(f"\nΔM = {delta_M:.2f}")

    # -----------------------------
    # 7. INTERPRETATION
    # -----------------------------
    if 1.0 <= delta_M <= 1.4:
        print("✅ Follows Bath's Law")
    else:
        print("❌ Does NOT follow Bath's Law")
else:
    print("Cannot apply Bath's Law: No aftershocks found to determine M_after.")